# 06. Embedding Visualization — t-SNE / UMAP
# 06. Embedding 시각화 — t-SNE / UMAP

**Goal / 목표**
- Extract embeddings from the Phase 0 backbone and visualize them in 2D
- Phase 0 backbone에서 embedding을 추출하고 2D로 투영하여 시각화한다
- Verify Grayspot Level boundary validity (PRD 3.2)
- Grayspot Level 경계의 타당성을 검증한다 (PRD 3.2)
- Compute ARI and Silhouette Score for refinement decision (PRD 3.2.3)
- 정제 완료 판단을 위해 ARI 와 Silhouette Score 를 계산한다 (PRD 3.2.3)

**Compatibility / 호환성**
- IMAGE_SIZE=128, cv2 loading — same as 02/03/05 notebooks
- IMAGE_SIZE=128, cv2 로딩 — 02/03/05 노트북과 동일
- Loads phase0_backbone_C.pt saved by 05_contrastive.ipynb
- 05_contrastive.ipynb 가 저장한 phase0_backbone_C.pt 를 로드한다
- Backbone structure matches SimCLRModel in 05_contrastive.ipynb
- 백본 구조가 05_contrastive.ipynb 의 SimCLRModel 과 일치한다

**Folder Structure / 폴더 구조**
```
CMYK_MAIN/
  data_set/
    labeled/{channel}/{level}/*.png
    models/phase0_backbone_C.pt  <- 05_contrastive.ipynb output
    embedding_viz/               <- this notebook output
    labels_v0.csv
```

**Execution Order / 실행 순서**
1. Import libraries
2. Path & settings
3. Load labels
4. Dataset & embedding extraction
5. Dimensionality reduction (t-SNE / UMAP)
6. Visualization (t-SNE by level, by color, subplots, UMAP)
7. Quantitative metrics (ARI, Silhouette)
8. Label-embedding mismatch analysis
9. Mismatch overlay
10. Metrics summary & Phase 1 decision
11. Metrics dashboard
12. Output summary

---
## 1. Import Libraries / 라이브러리 임포트

In [1]:
# 필요 패키지 설치 (최초 1회) / Install required packages (first time only)
# !pip install umap-learn scikit-learn plotly pandas numpy torch torchvision opencv-python

import json
import random
import sys
import warnings
import webbrowser
from pathlib import Path
from typing import Dict, List, Optional, Tuple

warnings.filterwarnings('ignore')

import cv2
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torchvision import models

from sklearn.manifold import TSNE
from sklearn.cluster import KMeans
from sklearn.metrics import adjusted_rand_score, silhouette_score
from sklearn.preprocessing import LabelEncoder

try:
    import umap
    UMAP_AVAILABLE = True
except ImportError:
    UMAP_AVAILABLE = False
    print('[Warning] umap-learn not installed -> UMAP skipped. pip install umap-learn')
    print('[경고] umap-learn 미설치 -> UMAP 건너뜀. pip install umap-learn')

import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

# Python version guard / 파이썬 버전 확인
assert sys.version_info[:2] == (3, 11), (
    f'Python 3.11.x required. Found: {sys.version}'
)

# Device setup — same logic as 02/03/05
# 디바이스 설정 — 02/03/05 와 동일한 방식
device = torch.device(
    'cuda' if torch.cuda.is_available() else
    'mps'  if torch.backends.mps.is_available() else
    'cpu'
)


def open_in_browser(path) -> None:
    """
    Open saved HTML file in default browser.
    저장된 HTML 파일을 기본 브라우저로 연다.
    Works on Windows and macOS via file:// URI.
    file:// URI 방식으로 Windows/macOS 모두 동작한다.
    """
    webbrowser.open(Path(path).resolve().as_uri())


print('Libraries loaded / 라이브러리 로드 완료')
print(f'PyTorch version / 버전: {torch.__version__}')
print(f'Device / 디바이스: {device}')
print(f'UMAP available / 사용 가능: {UMAP_AVAILABLE}')

Libraries loaded / 라이브러리 로드 완료
PyTorch version / 버전: 2.11.0
Device / 디바이스: mps
UMAP available / 사용 가능: True


---
## 2. Path & Settings / 경로 및 설정

> Edit only this cell — all paths propagate automatically.
> 이 셀만 수정하면 모든 경로가 자동으로 반영된다.

In [2]:
ROOT_DIR    = Path('../..').resolve()             # CMYK_MAIN/
LABELED_DIR = ROOT_DIR / 'data_set' / 'labeled'  # labeled/{channel}/{level}/*.png
MODELS_DIR  = ROOT_DIR / 'data_set' / 'models'   # phase0_backbone_C.pt etc.
LABELS_CSV  = ROOT_DIR / 'data_set' / 'labels_v0.csv'
OUTPUT_DIR  = ROOT_DIR / 'data_set' / 'embedding_viz'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# Checkpoint saved by 05_contrastive.ipynb (model.state_dict())
# 05_contrastive.ipynb 가 model.state_dict() 로 저장한 파일
#   MODELS_DIR / 'phase0_backbone_C.pt'
#   MODELS_DIR / 'phase0_backbone_Y.pt'
#   MODELS_DIR / 'baseline_C.pt'   <- Phase 2 checkpoint also works
CHECKPOINT = MODELS_DIR / 'phase0_backbone_C.pt'

# Data settings — must match 05_contrastive.ipynb
# 데이터 설정 — 05_contrastive.ipynb 와 반드시 일치해야 한다
CHANNELS      = ['Y', 'M', 'C', 'K']
NUM_LEVELS    = 6       # Level 0 ~ 5
IMAGE_SIZE    = 128     # same as 02/03/05 / 02/03/05 와 동일
BATCH_SIZE    = 32
BACKBONE_NAME = 'efficientnet_b0'  # must match 05_contrastive.ipynb
COLOR_COLUMNS = {'Y': 'Y', 'M': 'M', 'C': 'C', 'K': 'K'}

# t-SNE / UMAP parameters / 파라미터
TSNE_PARAMS = {'n_components': 2, 'perplexity': 30, 'n_iter': 1000, 'random_state': 42}
UMAP_PARAMS = {'n_components': 2, 'n_neighbors': 15, 'min_dist': 0.1, 'random_state': 42}

# Analysis thresholds / 분석 임계값 (PRD 3.2.3, 5.3.1)
ARI_THRESHOLD        = 0.6
MISMATCH_THRESHOLD   = 0.10
SILHOUETTE_THRESHOLD = 0.3

# Visualization colors / 시각화 색상
LEVEL_COLORS = ['#2ecc71', '#f1c40f', '#e67e22', '#e74c3c', '#9b59b6', '#1a1a2e']
CMYK_COLORS  = {'Y': '#f5e642', 'M': '#e91e8c', 'C': '#00b4d8', 'K': '#444444'}

# Validate paths / 경로 유효성 확인
assert LABELED_DIR.exists(), f'labeled/ not found: {LABELED_DIR}'
assert CHECKPOINT.exists(),  f'Checkpoint not found: {CHECKPOINT}'
assert LABELS_CSV.exists(),  f'labels_v0.csv not found: {LABELS_CSV}'

print(f'ROOT_DIR   : {ROOT_DIR}')
print(f'LABELED_DIR: {LABELED_DIR}')
print(f'CHECKPOINT : {CHECKPOINT}')
print(f'OUTPUT_DIR : {OUTPUT_DIR}')
print()
print('Labeled folder counts / 라벨링 폴더 샘플 수:')
for ch in CHANNELS:
    for lv in range(NUM_LEVELS):
        folder = LABELED_DIR / ch / str(lv)
        count  = len(list(folder.glob('*.png'))) if folder.exists() else 0
        print(f'  [{ch}] Level {lv}: {count}')

ROOT_DIR   : /Users/yangjin-hyeong/Desktop/CMYK_MAIN
LABELED_DIR: /Users/yangjin-hyeong/Desktop/CMYK_MAIN/data_set/labeled
CHECKPOINT : /Users/yangjin-hyeong/Desktop/CMYK_MAIN/data_set/models/phase0_backbone_C.pt
OUTPUT_DIR : /Users/yangjin-hyeong/Desktop/CMYK_MAIN/data_set/embedding_viz

Labeled folder counts / 라벨링 폴더 샘플 수:
  [Y] Level 0: 7
  [Y] Level 1: 23
  [Y] Level 2: 39
  [Y] Level 3: 38
  [Y] Level 4: 5
  [Y] Level 5: 0
  [M] Level 0: 14
  [M] Level 1: 137
  [M] Level 2: 228
  [M] Level 3: 472
  [M] Level 4: 200
  [M] Level 5: 73
  [C] Level 0: 0
  [C] Level 1: 16
  [C] Level 2: 34
  [C] Level 3: 89
  [C] Level 4: 32
  [C] Level 5: 2
  [K] Level 0: 5
  [K] Level 1: 16
  [K] Level 2: 71
  [K] Level 3: 231
  [K] Level 4: 121
  [K] Level 5: 2


---
## 3. Load Labels / 라벨 로드

Filename convention: scan_{num}_{color}_{num}.png
파일명 규칙: scan_{번호}_{색상}_{번호}.png

Each file exists only in the folder matching its color.
각 파일은 파일명의 색상 폴더에만 존재한다.
e.g. scan_001_C_0007.png -> labeled/C/{level}/ only

In [3]:
def extract_color_from_filename(fname: str) -> Optional[str]:
    """
    Extract color code from filename.
    파일명에서 색상 코드를 추출한다.
    e.g. 'scan_001_C_0007.png' -> 'C'
    """
    for part in Path(fname).stem.split('_'):
        if part in ('Y', 'M', 'C', 'K'):
            return part
    return None


def load_labels(labels_csv: Path, color_columns: dict) -> pd.DataFrame:
    """
    Load label CSV and convert to long-format DataFrame.
    라벨 CSV를 로드하여 long-format DataFrame으로 변환한다.

    Only the label matching the color in the filename is kept.
    파일명의 색상과 일치하는 라벨만 유효하다.
    """
    df = pd.read_csv(labels_csv)
    print(f'CSV loaded / 로드: {len(df)} rows, columns: {list(df.columns)}')

    records = []
    skipped = 0
    for _, row in df.iterrows():
        color = extract_color_from_filename(str(row['filename']))
        if color is None:
            skipped += 1
            continue
        col = color_columns.get(color)
        if col is None or col not in df.columns:
            skipped += 1
            continue
        records.append({
            'filename'  : row['filename'],
            'color'     : color,
            'level'     : int(row[col]),
            'confidence': row.get('confidence', 'certain'),
        })

    long_df = pd.DataFrame(records)
    print(f'Long-format rows / 변환 결과: {len(long_df)}  (skipped / 제외: {skipped})')
    print(long_df.groupby(['color', 'level']).size().unstack(fill_value=0))
    return long_df


df_labels = load_labels(LABELS_CSV, COLOR_COLUMNS)
df_labels['level_str'] = df_labels['level'].astype(str)

print(f'\nTotal samples / 총 샘플: {len(df_labels)}')
df_labels.head()

CSV loaded / 로드: 1855 rows, columns: ['filename', 'C', 'M', 'Y', 'K']
Long-format rows / 변환 결과: 1855  (skipped / 제외: 0)
level   0    1    2    3    4   5
color                            
C       0   16   34   89   32   2
K       5   16   71  231  121   2
M      14  137  228  472  200  73
Y       7   23   39   38    5   0

Total samples / 총 샘플: 1855


,filename,color,level,confidence,level_str
0,scan_001_C_0007.png,C,2,certain,2
1,scan_002_C_0019.png,C,1,certain,1
2,scan_002_C_0025.png,C,1,certain,1
3,scan_002_C_0031.png,C,3,certain,3
4,scan_002_K_0008.png,K,3,certain,3


---
## 4. Dataset & Embedding Extraction / 데이터셋 & Embedding 추출

Image loading follows 05_contrastive.ipynb — cv2, same preprocessing.
05_contrastive.ipynb 방식을 따른다 — cv2, 동일한 전처리.

Backbone is loaded from phase0_backbone_C.pt (SimCLRModel.state_dict()).
backbone 키만 필터링하여 로드한다. (05_contrastive.ipynb 의 SimCLRModel.state_dict() 에서 backbone. 키 추출)

In [4]:
class EmbeddingDataset(Dataset):
    """
    Dataset for embedding extraction.
    Embedding 추출 전용 Dataset.

    Loads images with cv2, same as 02/03/05.
    cv2 로 이미지를 로드한다. 02/03/05 와 동일.

    Folder structure / 폴더 구조:
        labeled/{color}/{level}/{filename}
    """

    def __init__(self, df: pd.DataFrame, patch_dir: Path, image_size: int):
        self.df         = df.reset_index(drop=True)
        self.patch_dir  = Path(patch_dir)
        self.image_size = image_size

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row   = self.df.iloc[idx]
        color = row['color']
        fname = row['filename']
        level = int(row['level'])

        img_path = self.patch_dir / color / str(level) / fname

        if not img_path.exists():
            raise FileNotFoundError(
                f'Image not found / 이미지 없음: {img_path}'
            )

        # cv2 loading — same as 02/03/05 / 02/03/05 와 동일한 cv2 로딩
        img = cv2.imread(str(img_path))
        img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        img = cv2.resize(img, (self.image_size, self.image_size))
        img = img.astype(np.float32) / 255.0  # [0,1] normalize

        # (H, W, 3) -> (3, H, W) tensor
        tensor = torch.tensor(img).permute(2, 0, 1).float()

        return tensor, idx  # return original index for order tracking


def load_backbone_from_checkpoint(
    backbone_name : str,
    checkpoint    : Path,
    device        : torch.device,
) -> Tuple[nn.Module, int]:
    """
    Load backbone from checkpoint saved by 05_contrastive.ipynb.
    05_contrastive.ipynb 가 저장한 체크포인트에서 backbone 을 로드한다.

    05_contrastive.ipynb saves model.state_dict() of SimCLRModel.
    SimCLRModel has keys: backbone.*, head.*
    We extract only backbone.* keys.

    05_contrastive.ipynb 는 SimCLRModel 의 model.state_dict() 를 저장한다.
    키 구조: backbone.*, head.*
    backbone.* 키만 추출하여 사용한다.
    """
    # Build backbone — same logic as 02/03/05
    # 백본 구성 — 02/03/05 와 동일한 방식
    if backbone_name == 'efficientnet_b0':
        from torchvision.models import EfficientNet_B0_Weights
        backbone    = models.efficientnet_b0(weights=EfficientNet_B0_Weights.DEFAULT)
        feature_dim = backbone.classifier[1].in_features  # 1280
        backbone.classifier = nn.Identity()

    elif backbone_name == 'resnet50':
        from torchvision.models import ResNet50_Weights
        backbone    = models.resnet50(weights=ResNet50_Weights.DEFAULT)
        feature_dim = backbone.fc.in_features  # 2048
        backbone.fc = nn.Identity()

    else:
        raise ValueError(f'Unsupported backbone: {backbone_name}')

    # Load checkpoint — filter backbone.* keys only
    # 체크포인트 로드 — backbone.* 키만 필터링
    state = torch.load(str(checkpoint), map_location='cpu')
    if isinstance(state, dict) and 'model_state_dict' in state:
        state = state['model_state_dict']

    # Filter backbone keys — works for both SimCLRModel and GrayspotModel
    # SimCLRModel 과 GrayspotModel 모두 backbone.* 키 구조를 사용하므로 호환 가능
    backbone_state = {
        k.replace('backbone.', ''): v
        for k, v in state.items()
        if k.startswith('backbone.')
    }

    if backbone_state:
        backbone.load_state_dict(backbone_state, strict=False)
        print(f'Backbone keys loaded / 로드: {len(backbone_state)} keys from {checkpoint.name}')
    else:
        # Fallback: try loading full state (no backbone. prefix)
        # 폴백: backbone. 키 없으면 전체 state 시도
        backbone.load_state_dict(state, strict=False)
        print(f'Full state loaded (no backbone. prefix) / 전체 state 로드: {checkpoint.name}')

    backbone.eval()
    return backbone.to(device), feature_dim


@torch.no_grad()
def extract_embeddings(
    backbone   : nn.Module,
    dataloader : DataLoader,
    device     : torch.device,
) -> np.ndarray:
    """
    Extract embedding vectors from backbone.
    백본에서 embedding 벡터를 일괄 추출한다.

    Returns / 반환:
        embeddings: (N, feature_dim) numpy array
    """
    backbone.eval()
    all_embeds = []
    for batch_tensors, _ in dataloader:
        batch_tensors = batch_tensors.to(device)
        feats         = backbone(batch_tensors)  # (B, feature_dim)
        all_embeds.append(feats.cpu().numpy())
    return np.concatenate(all_embeds, axis=0)


# Load backbone / 백본 로드
backbone, feature_dim = load_backbone_from_checkpoint(BACKBONE_NAME, CHECKPOINT, device)
print(f'Backbone: {BACKBONE_NAME}  feature_dim: {feature_dim}')

# Build dataset and extract embeddings / 데이터셋 구성 후 embedding 추출
dataset = EmbeddingDataset(df_labels, LABELED_DIR, IMAGE_SIZE)
loader  = DataLoader(
    dataset,
    batch_size  = BATCH_SIZE,
    shuffle     = False,
    num_workers = 0,  # 0 = safe on Windows/macOS
    pin_memory  = (device.type == 'cuda'),
)

print(f'\nExtracting embeddings / Embedding 추출 중... ({len(dataset)} samples)')
embeddings = extract_embeddings(backbone, loader, device)
print(f'Embedding shape: {embeddings.shape}  (samples x dim / 샘플 수 x 차원)')

Backbone keys loaded / 로드: 358 keys from phase0_backbone_C.pt
Backbone: efficientnet_b0  feature_dim: 1280

Extracting embeddings / Embedding 추출 중... (1855 samples)
Embedding shape: (1855, 1280)  (samples x dim / 샘플 수 x 차원)


---
## 5. Dimensionality Reduction / 차원 축소 (t-SNE / UMAP)

In [5]:
def run_tsne(embeddings: np.ndarray, params: dict) -> np.ndarray:
    """
    Run t-SNE 2D projection.
    t-SNE 2D 투영을 실행한다.
    """
    n          = len(embeddings)
    perplexity = min(params['perplexity'], n // 4)
    print(f'[t-SNE] n_samples={n}, perplexity={perplexity}, n_iter={params["n_iter"]}')

    tsne = TSNE(
        n_components=params['n_components'],
        perplexity=perplexity,
        max_iter=params['n_iter'],       # n_iter -> max_iter (scikit-learn 1.5+)
        random_state=params['random_state'],
        init='pca',
        learning_rate='auto',
        verbose=0,
    )
    proj = tsne.fit_transform(embeddings)
    print(f'[t-SNE done / 완료] KL Divergence: {tsne.kl_divergence_:.4f}')
    return proj


def run_umap(embeddings: np.ndarray, params: dict) -> Optional[np.ndarray]:
    """
    Run UMAP 2D projection. Returns None if umap-learn is not installed.
    UMAP 2D 투영. umap-learn 미설치 시 None 반환.
    """
    if not UMAP_AVAILABLE:
        return None
    print(f'[UMAP] n_neighbors={params["n_neighbors"]}, min_dist={params["min_dist"]}')
    reducer = umap.UMAP(
        n_components=params['n_components'],
        n_neighbors=params['n_neighbors'],
        min_dist=params['min_dist'],
        random_state=params['random_state'],
        verbose=False,
    )
    proj = reducer.fit_transform(embeddings)
    print('[UMAP done / 완료]')
    return proj


print('=' * 50)
print('Running t-SNE / t-SNE 실행 중... (may take several minutes / 수분 소요 가능)')
tsne_proj = run_tsne(embeddings, TSNE_PARAMS)

print('=' * 50)
print('Running UMAP / UMAP 실행 중...')
umap_proj = run_umap(embeddings, UMAP_PARAMS)

# Add projection coordinates to DataFrame / 투영 결과를 DataFrame에 추가
df_viz           = df_labels.copy()
df_viz['tsne_x'] = tsne_proj[:, 0]
df_viz['tsne_y'] = tsne_proj[:, 1]

if umap_proj is not None:
    df_viz['umap_x'] = umap_proj[:, 0]
    df_viz['umap_y'] = umap_proj[:, 1]

print('\nDimensionality reduction complete / 차원 축소 완료')
df_viz.head()

Running t-SNE / t-SNE 실행 중... (may take several minutes / 수분 소요 가능)
[t-SNE] n_samples=1855, perplexity=30, n_iter=1000
[t-SNE done / 완료] KL Divergence: 0.8867
Running UMAP / UMAP 실행 중...
[UMAP] n_neighbors=15, min_dist=0.1
[UMAP done / 완료]

Dimensionality reduction complete / 차원 축소 완료


,filename,color,level,confidence,level_str,tsne_x,tsne_y,umap_x,umap_y
0,scan_001_C_0007.png,C,2,certain,2,-13.589012,31.850065,2.668008,7.185129
1,scan_002_C_0019.png,C,1,certain,1,-35.239616,-16.816206,-2.445998,5.595031
2,scan_002_C_0025.png,C,1,certain,1,-35.307930,-16.851025,-2.480922,5.556762
3,scan_002_C_0031.png,C,3,certain,3,-35.305271,-16.967953,-2.495719,5.618678
4,scan_002_K_0008.png,K,3,certain,3,-44.930912,-9.062561,-2.259384,6.581984


---
## 6. Visualization / 시각화

### 6.1 t-SNE — Level-colored / Level별 색상 코딩

In [6]:
def plot_projection_by_level(
    df           : pd.DataFrame,
    x_col        : str,
    y_col        : str,
    method_name  : str,
    level_colors : List[str],
    output_path  : Optional[str] = None,
) -> go.Figure:
    """
    Level-colored scatter plot.
    Level별 색상 코딩 scatter plot.
    """
    color_map = {str(i): c for i, c in enumerate(level_colors)}

    fig = px.scatter(
        df, x=x_col, y=y_col,
        color='level_str',
        color_discrete_map=color_map,
        symbol='color',
        hover_data=['filename', 'color', 'level', 'confidence'],
        category_orders={'level_str': [str(i) for i in range(6)]},
        title=f'{method_name} — Embedding Distribution by Level / Level별 Embedding 분포',
        labels={'level_str': 'Level (0~5)', 'color': 'CMYK Channel'},
        template='plotly_dark',
        width=900, height=650,
    )
    fig.update_traces(marker=dict(size=7, opacity=0.85,
                                  line=dict(width=0.5, color='rgba(255,255,255,0.3)')))
    fig.update_layout(
        legend_title_text='Level',
        font=dict(family='Segoe UI', size=12),
        title_font=dict(size=15),
        margin=dict(l=40, r=40, t=60, b=40),
    )

    if output_path:
        fig.write_html(output_path, include_plotlyjs='cdn')
        print(f'Saved / 저장: {output_path}')

    return fig


fig_tsne_level = plot_projection_by_level(
    df_viz, 'tsne_x', 'tsne_y',
    method_name='t-SNE',
    level_colors=LEVEL_COLORS,
    output_path=str(OUTPUT_DIR / 'tsne_by_level.html'),
)
open_in_browser(OUTPUT_DIR / 'tsne_by_level.html')

Saved / 저장: /Users/yangjin-hyeong/Desktop/CMYK_MAIN/data_set/embedding_viz/tsne_by_level.html


### 6.2 t-SNE — Per-CMYK Distribution / CMYK 색상별 분포 (PRD 5.6.4)

In [7]:
def plot_projection_by_color(
    df          : pd.DataFrame,
    x_col       : str,
    y_col       : str,
    method_name : str,
    cmyk_colors : Dict[str, str],
    output_path : Optional[str] = None,
) -> go.Figure:
    """
    CMYK channel-colored scatter plot.
    CMYK 채널별 색상 코딩 scatter plot.
    """
    fig = px.scatter(
        df, x=x_col, y=y_col,
        color='color',
        color_discrete_map=cmyk_colors,
        size='level', size_max=18,
        hover_data=['filename', 'color', 'level', 'confidence'],
        title=f'{method_name} — Distribution by CMYK Channel / CMYK 채널별 분포 (marker size=Level)',
        labels={'color': 'CMYK Channel'},
        template='plotly_dark',
        width=900, height=650,
    )
    fig.update_traces(marker=dict(opacity=0.75,
                                  line=dict(width=0.5, color='rgba(255,255,255,0.2)')))
    fig.update_layout(
        font=dict(family='Segoe UI', size=12),
        title_font=dict(size=15),
        margin=dict(l=40, r=40, t=60, b=40),
    )

    if output_path:
        fig.write_html(output_path, include_plotlyjs='cdn')
        print(f'Saved / 저장: {output_path}')

    return fig


fig_tsne_color = plot_projection_by_color(
    df_viz, 'tsne_x', 'tsne_y',
    method_name='t-SNE',
    cmyk_colors=CMYK_COLORS,
    output_path=str(OUTPUT_DIR / 'tsne_by_color.html'),
)
open_in_browser(OUTPUT_DIR / 'tsne_by_color.html')

Saved / 저장: /Users/yangjin-hyeong/Desktop/CMYK_MAIN/data_set/embedding_viz/tsne_by_color.html


### 6.3 t-SNE — Per-color 4-panel Subplot / CMYK 4분할 서브플롯

In [8]:
def plot_per_color_subplot(
    df           : pd.DataFrame,
    x_col        : str,
    y_col        : str,
    method_name  : str,
    level_colors : List[str],
    output_path  : Optional[str] = None,
) -> go.Figure:
    """
    CMYK 4-panel subplot comparing Level distribution per channel.
    CMYK 4채널을 2x2 서브플롯으로 Level 분포 비교.
    """
    colors_order = ['Y', 'M', 'C', 'K']
    color_titles = {'Y': 'Yellow', 'M': 'Magenta', 'C': 'Cyan', 'K': 'Black'}

    fig = make_subplots(
        rows=2, cols=2,
        subplot_titles=[color_titles[c] for c in colors_order],
        horizontal_spacing=0.08,
        vertical_spacing=0.12,
    )

    level_color_map = {i: level_colors[i] for i in range(6)}
    shown_in_legend = set()

    for idx, color_code in enumerate(colors_order):
        row  = idx // 2 + 1
        col  = idx % 2  + 1
        df_c = df[df['color'] == color_code]

        for level in range(6):
            df_lv       = df_c[df_c['level'] == level]
            if len(df_lv) == 0:
                continue
            show_legend = level not in shown_in_legend
            fig.add_trace(
                go.Scatter(
                    x=df_lv[x_col], y=df_lv[y_col],
                    mode='markers',
                    name=f'Level {level}',
                    legendgroup=f'Level {level}',
                    showlegend=show_legend,
                    marker=dict(
                        color=level_color_map[level],
                        size=6, opacity=0.8,
                        line=dict(width=0.3, color='rgba(255,255,255,0.2)'),
                    ),
                    text=df_lv['filename'],
                    hovertemplate=(
                        f'Color: {color_code}<br>'
                        'Level: %{marker.color}<br>'
                        'File: %{text}<extra></extra>'
                    ),
                ),
                row=row, col=col,
            )
            shown_in_legend.add(level)

    fig.update_layout(
        title=f'{method_name} — Level Distribution by CMYK Channel (4-panel) / CMYK 채널별 Level 분포 (4분할)',
        title_font=dict(size=15),
        template='plotly_dark',
        font=dict(family='Segoe UI', size=11),
        width=1100, height=800,
        margin=dict(l=40, r=40, t=80, b=40),
    )

    if output_path:
        fig.write_html(output_path, include_plotlyjs='cdn')
        print(f'Saved / 저장: {output_path}')

    return fig


subplot_path = str(OUTPUT_DIR / 'tsne_per_color_subplot.html')
plot_per_color_subplot(
    df_viz, 'tsne_x', 'tsne_y',
    method_name='t-SNE',
    level_colors=LEVEL_COLORS,
    output_path=subplot_path,
)
open_in_browser(OUTPUT_DIR / 'tsne_per_color_subplot.html')

Saved / 저장: /Users/yangjin-hyeong/Desktop/CMYK_MAIN/data_set/embedding_viz/tsne_per_color_subplot.html


### 6.4 UMAP (if installed / 설치된 경우)

In [9]:
if umap_proj is not None:
    plot_projection_by_level(
        df_viz, 'umap_x', 'umap_y',
        method_name='UMAP',
        level_colors=LEVEL_COLORS,
        output_path=str(OUTPUT_DIR / 'umap_by_level.html'),
    )
    open_in_browser(OUTPUT_DIR / 'umap_by_level.html')
else:
    print('UMAP not available — skipping / UMAP 미설치 — 건너뜀')

Saved / 저장: /Users/yangjin-hyeong/Desktop/CMYK_MAIN/data_set/embedding_viz/umap_by_level.html


---
## 7. Quantitative Metrics / 정량 지표

### 7.1 ARI & Silhouette Score (PRD 3.2.3, 5.3.1)

In [10]:
def compute_ari(
    embeddings   : np.ndarray,
    true_labels  : np.ndarray,
    n_clusters   : int = 6,
    random_state : int = 42,
) -> float:
    """
    Apply k-means to embeddings, compute ARI against Level labels.
    embedding에 k-means 적용 후 Level 라벨과의 ARI 산출.
    PRD 3.2.3: ARI >= 0.6 -> Phase 1 refinement criteria met
    """
    km             = KMeans(n_clusters=n_clusters, random_state=random_state, n_init=10)
    cluster_labels = km.fit_predict(embeddings)
    return float(adjusted_rand_score(true_labels, cluster_labels))


def compute_silhouette(embeddings: np.ndarray, labels: np.ndarray) -> float:
    """
    Compute Silhouette Score.
    Silhouette Score 산출.
    """
    if len(set(labels)) < 2:
        return 0.0
    n = len(embeddings)
    if n > 5000:
        idx        = np.random.choice(n, 5000, replace=False)
        embeddings = embeddings[idx]
        labels     = labels[idx]
    return float(silhouette_score(embeddings, labels, sample_size=min(n, 3000), random_state=42))


print('=' * 50)
print('Computing quantitative metrics / 정량 지표 계산 중...')

level_labels     = df_viz['level'].values
color_labels_enc = LabelEncoder().fit_transform(df_viz['color'].values)

ari_all   = compute_ari(embeddings, level_labels)
sil_level = compute_silhouette(embeddings, level_labels)
sil_color = compute_silhouette(embeddings, color_labels_enc)

print(f'\n[Global] ARI (Level vs k-means)   : {ari_all:.4f}')
print(f'  Target >= {ARI_THRESHOLD}  |  {"Met / 충족" if ari_all >= ARI_THRESHOLD else "Not met / 미달 -- Phase 1 refinement needed"}')
print(f'[Global] Silhouette (Level)       : {sil_level:.4f}')
print(f'[Global] Silhouette (CMYK Color)  : {sil_color:.4f}')
print(f'  Independent model threshold >= {SILHOUETTE_THRESHOLD}  |  '
      f'{"Per-color model recommended / 색상별 독립 모델 검토 권장" if sil_color >= SILHOUETTE_THRESHOLD else "Shared model suitable / 공유 모델 적합"}')

ari_by_color = {}
sil_by_color = {}
print('\n[Per-color metrics / 색상별 지표]')
for color_code in ['Y', 'M', 'C', 'K']:
    mask  = df_viz['color'].values == color_code
    emb_c = embeddings[mask]
    lv_c  = level_labels[mask]
    if len(emb_c) < 10:
        continue
    ari_c                    = compute_ari(emb_c, lv_c)
    sil_c                    = compute_silhouette(emb_c, lv_c)
    ari_by_color[color_code] = ari_c
    sil_by_color[color_code] = sil_c
    print(f'  {color_code}: ARI={ari_c:.4f}  Silhouette(Level)={sil_c:.4f}')

Computing quantitative metrics / 정량 지표 계산 중...

[Global] ARI (Level vs k-means)   : 0.0508
  Target >= 0.6  |  Not met / 미달 -- Phase 1 refinement needed
[Global] Silhouette (Level)       : -0.0183
[Global] Silhouette (CMYK Color)  : 0.1815
  Independent model threshold >= 0.3  |  Shared model suitable / 공유 모델 적합

[Per-color metrics / 색상별 지표]
  Y: ARI=0.0991  Silhouette(Level)=-0.0664
  M: ARI=0.0930  Silhouette(Level)=-0.0535
  C: ARI=0.0302  Silhouette(Level)=-0.0475
  K: ARI=0.0389  Silhouette(Level)=-0.0462


---
## 8. Label-Embedding Mismatch Analysis / 라벨-Embedding 불일치 분석 (PRD 5.6.4)

In [11]:
def find_mismatch_samples(
    df           : pd.DataFrame,
    embeddings   : np.ndarray,
    n_clusters   : int = 6,
    random_state : int = 42,
) -> pd.DataFrame:
    """
    Find samples where k-means cluster and Level label do not match.
    k-means 클러스터 할당과 Level 라벨이 불일치하는 샘플을 추출한다.

    The representative Level of each cluster is the mode.
    각 클러스터의 대표 Level = 최빈값.
    Samples in the cluster that differ from the mode = mismatch.
    클러스터 내 최빈값이 아닌 샘플 = 불일치 샘플.
    """
    km          = KMeans(n_clusters=n_clusters, random_state=random_state, n_init=10)
    cluster_ids = km.fit_predict(embeddings)

    df               = df.copy()
    df['cluster_id'] = cluster_ids

    cluster_mode = (
        df.groupby('cluster_id')['level']
        .agg(lambda x: x.mode().iloc[0])
        .rename('cluster_level')
    )
    df = df.join(cluster_mode, on='cluster_id')

    mismatch               = df[df['level'] != df['cluster_level']].copy()
    mismatch['level_diff'] = (mismatch['level'] - mismatch['cluster_level']).abs()
    mismatch               = mismatch.sort_values('level_diff', ascending=False)

    return mismatch


df_mismatch    = find_mismatch_samples(df_viz, embeddings)
mismatch_ratio = len(df_mismatch) / len(df_viz)

print(f'Mismatch count / 불일치 수: {len(df_mismatch)} / {len(df_viz)} ({mismatch_ratio * 100:.1f}%)')
print(f'  Threshold <= {MISMATCH_THRESHOLD * 100:.0f}%  |  '
      f'{"Met / 충족" if mismatch_ratio <= MISMATCH_THRESHOLD else "Exceeded / 초과 -- label refinement needed / 라벨 정제 필요"}')

print('\n[Per-color mismatch / 색상별 불일치]')
print(df_mismatch.groupby('color').size().to_string())

mismatch_path = OUTPUT_DIR / 'mismatch_samples.csv'
df_mismatch.to_csv(mismatch_path, index=False, encoding='utf-8-sig')
print(f'\nSaved / 저장: {mismatch_path}')
df_mismatch.head(10)

Mismatch count / 불일치 수: 997 / 1855 (53.7%)
  Threshold <= 10%  |  Exceeded / 초과 -- label refinement needed / 라벨 정제 필요

[Per-color mismatch / 색상별 불일치]
color
C     84
K    215
M    624
Y     74

Saved / 저장: /Users/yangjin-hyeong/Desktop/CMYK_MAIN/data_set/embedding_viz/mismatch_samples.csv


,filename,color,level,confidence,level_str,tsne_x,tsne_y,umap_x,umap_y,cluster_id,cluster_level,level_diff
946,scan_075_M_0005.png,M,5,certain,5,3.266859,9.314154,-2.795872,23.540400,3,1,4
795,scan_064_M_0005.png,M,5,certain,5,5.251807,7.394985,-2.729613,23.531893,3,1,4
848,scan_069_M_0002.png,M,5,certain,5,3.686601,5.792111,-2.702996,23.528996,3,1,4
945,scan_075_M_0004.png,M,5,certain,5,3.270028,9.183133,-2.786809,23.540485,3,1,4
910,scan_073_M_0004.png,M,5,certain,5,3.318535,8.549819,-2.781107,23.550406,3,1,4
1121,scan_084_M_0005.png,M,5,certain,5,4.504695,8.436893,-2.862061,23.463909,3,1,4
1120,scan_084_M_0004.png,M,5,certain,5,3.987979,6.513629,-2.697932,23.557486,3,1,4
1008,scan_078_M_0002.png,M,5,certain,5,4.756086,7.895011,-2.755514,23.515627,3,1,4
1628,scan_151_M_0039.png,M,5,certain,5,7.075450,35.573036,0.509262,16.554235,3,1,4
1069,scan_081_M_0005.png,M,5,certain,5,4.681200,5.591432,-2.703086,23.559660,3,1,4


---
## 9. Mismatch Overlay / 불일치 샘플 시각화

In [12]:
def plot_mismatch_overlay(
    df_all      : pd.DataFrame,
    df_mismatch : pd.DataFrame,
    x_col       : str,
    y_col       : str,
    level_colors: List[str],
    output_path : Optional[str] = None,
) -> go.Figure:
    """
    t-SNE scatter with mismatch samples highlighted as star markers.
    불일치 샘플을 별 마커로 강조한 t-SNE scatter plot.
    """
    color_map = {str(i): c for i, c in enumerate(level_colors)}

    fig = px.scatter(
        df_all, x=x_col, y=y_col,
        color='level_str',
        color_discrete_map=color_map,
        title='t-SNE — Mismatch Samples Highlighted / 불일치 샘플 강조',
        template='plotly_dark',
        hover_data=['filename', 'color', 'level'],
        width=900, height=650,
    )
    fig.update_traces(marker=dict(size=6, opacity=0.5))

    if len(df_mismatch) > 0 and x_col in df_mismatch.columns:
        fig.add_trace(go.Scatter(
            x=df_mismatch[x_col],
            y=df_mismatch[y_col],
            mode='markers',
            name='Mismatch / 불일치',
            marker=dict(
                symbol='star', size=14, color='#ff7aa2',
                line=dict(width=1, color='white'), opacity=0.95,
            ),
            text=df_mismatch.apply(
                lambda r: f"{r['filename']}<br>Actual: {r['level']}  Cluster: {r['cluster_level']}",
                axis=1,
            ),
            hovertemplate='%{text}<extra></extra>',
        ))

    fig.update_layout(
        font=dict(family='Segoe UI', size=12),
        margin=dict(l=40, r=40, t=60, b=40),
    )

    if output_path:
        fig.write_html(output_path, include_plotlyjs='cdn')
        print(f'Saved / 저장: {output_path}')

    return fig


plot_mismatch_overlay(
    df_viz, df_mismatch.copy(),
    'tsne_x', 'tsne_y',
    level_colors=LEVEL_COLORS,
    output_path=str(OUTPUT_DIR / 'tsne_mismatch_overlay.html'),
)
open_in_browser(OUTPUT_DIR / 'tsne_mismatch_overlay.html')

Saved / 저장: /Users/yangjin-hyeong/Desktop/CMYK_MAIN/data_set/embedding_viz/tsne_mismatch_overlay.html


---
## 10. Metrics Summary & Phase 1 Decision / 지표 요약 및 Phase 1 판단 (PRD 3.2.3)

In [13]:
# Aggregate metrics / 지표 집계
metrics_summary = {
    'meta': {
        'n_samples' : int(len(df_viz)),
        'backbone'  : BACKBONE_NAME,
        'checkpoint': str(CHECKPOINT),
        'image_size': IMAGE_SIZE,
    },
    'global': {
        'ARI'                          : round(ari_all,   4),
        'ARI_threshold'                : ARI_THRESHOLD,
        'ARI_pass'                     : bool(ari_all >= ARI_THRESHOLD),
        'silhouette_level'             : round(sil_level, 4),
        'silhouette_color'             : round(sil_color, 4),
        'silhouette_color_threshold'   : SILHOUETTE_THRESHOLD,
        'independent_model_recommended': bool(sil_color >= SILHOUETTE_THRESHOLD),
        'mismatch_count'               : int(len(df_mismatch)),
        'mismatch_ratio'               : round(mismatch_ratio, 4),
        'mismatch_threshold'           : MISMATCH_THRESHOLD,
        'mismatch_pass'                : bool(mismatch_ratio <= MISMATCH_THRESHOLD),
    },
    'by_color': {
        color: {
            'ARI'       : round(ari_by_color.get(color, -1), 4),
            'silhouette': round(sil_by_color.get(color, -1), 4),
        }
        for color in ['Y', 'M', 'C', 'K']
    },
}

metrics_path = OUTPUT_DIR / 'metrics_summary.json'
with open(metrics_path, 'w', encoding='utf-8') as f:
    json.dump(metrics_summary, f, ensure_ascii=False, indent=2)
print(f'Saved / 저장: {metrics_path}')

# Phase 1 decision output / Phase 1 판단 출력
print('\n' + '=' * 60)
print('Phase 1 Refinement Completion Criteria / 정제 완료 판단 (PRD 3.2.3)')
print('=' * 60)

cond_ari      = metrics_summary['global']['ARI_pass']
cond_mismatch = metrics_summary['global']['mismatch_pass']

print(f'  [1] ARI >= {ARI_THRESHOLD}       : {ari_all:.4f}  ->  {"Met / 충족" if cond_ari else "Not met / 미달"}')
print(f'  [2] Mismatch <= {MISMATCH_THRESHOLD*100:.0f}% : {mismatch_ratio*100:.1f}%  ->  {"Met / 충족" if cond_mismatch else "Not met / 미달"}')
print()

if cond_ari and cond_mismatch:
    print('  [PASS] Phase 1 criteria met -> Ready for Phase 2 / Phase 2 진입 가능')
else:
    print('  [FAIL] Criteria not met -> Review mismatch_samples.csv and refine labels')
    print('         기준 미달 -> mismatch_samples.csv 검토 후 라벨 재정제 필요')

print()
independent = metrics_summary['global']['independent_model_recommended']
print(f'  [3] Per-color independent model check (PRD 5.3.1)')
print(f'      Silhouette(CMYK) = {sil_color:.4f}  {">=" if independent else "<"} {SILHOUETTE_THRESHOLD}')
print(f'      -> {"Per-color independent model recommended / 색상별 독립 모델 검토 권장" if independent else "Shared model suitable / 공유 모델 적합"}')

Saved / 저장: /Users/yangjin-hyeong/Desktop/CMYK_MAIN/data_set/embedding_viz/metrics_summary.json

Phase 1 Refinement Completion Criteria / 정제 완료 판단 (PRD 3.2.3)
  [1] ARI >= 0.6       : 0.0508  ->  Not met / 미달
  [2] Mismatch <= 10% : 53.7%  ->  Not met / 미달

  [FAIL] Criteria not met -> Review mismatch_samples.csv and refine labels
         기준 미달 -> mismatch_samples.csv 검토 후 라벨 재정제 필요

  [3] Per-color independent model check (PRD 5.3.1)
      Silhouette(CMYK) = 0.1815  < 0.3
      -> Shared model suitable / 공유 모델 적합


---
## 11. Metrics Dashboard / 지표 시각화 대시보드 (HTML)

In [14]:
def plot_metrics_dashboard(
    metrics     : dict,
    output_path : Optional[str] = None,
) -> go.Figure:
    """
    Gauge + Bar dashboard for ARI, mismatch ratio, Silhouette.
    ARI, 불일치 비율, Silhouette 를 Gauge + Bar 로 시각화.
    """
    g            = metrics['global']
    colors_order = ['Y', 'M', 'C', 'K']

    fig = make_subplots(
        rows=2, cols=3,
        specs=[
            [{'type': 'indicator'}, {'type': 'indicator'}, {'type': 'indicator'}],
            [{'type': 'bar', 'colspan': 3}, None, None],
        ],
        subplot_titles=[
            'ARI (Level vs k-means)',
            'Mismatch Ratio / 불일치 비율',
            'Silhouette (CMYK Color)',
            'Per-color ARI / 색상별 ARI',
        ],
        vertical_spacing=0.15,
    )

    fig.add_trace(go.Indicator(
        mode='gauge+number', value=g['ARI'],
        number={'font': {'size': 30}},
        gauge={
            'axis': {'range': [0, 1]}, 'bar': {'color': '#50e3c2'},
            'threshold': {'line': {'color': '#ff7aa2', 'width': 3}, 'value': g['ARI_threshold']},
            'bgcolor': '#0b1220',
        },
        title={'text': f'Target >= {g["ARI_threshold"]}'},
    ), row=1, col=1)

    fig.add_trace(go.Indicator(
        mode='gauge+number', value=round(g['mismatch_ratio'] * 100, 1),
        number={'suffix': '%', 'font': {'size': 30}},
        gauge={
            'axis': {'range': [0, 50]}, 'bar': {'color': '#66d9ff'},
            'threshold': {'line': {'color': '#ff7aa2', 'width': 3},
                          'value': g['mismatch_threshold'] * 100},
            'bgcolor': '#0b1220',
        },
        title={'text': f'Target <= {g["mismatch_threshold"]*100:.0f}%'},
    ), row=1, col=2)

    fig.add_trace(go.Indicator(
        mode='gauge+number', value=g['silhouette_color'],
        number={'font': {'size': 30}},
        gauge={
            'axis': {'range': [-1, 1]}, 'bar': {'color': '#c792ea'},
            'threshold': {'line': {'color': '#ffb347', 'width': 3},
                          'value': g['silhouette_color_threshold']},
            'bgcolor': '#0b1220',
        },
        title={'text': f'Independent model threshold >= {g["silhouette_color_threshold"]}'},
    ), row=1, col=3)

    by_color = metrics['by_color']
    fig.add_trace(go.Bar(
        x=colors_order,
        y=[by_color[c]['ARI'] for c in colors_order],
        marker_color=['#f5e642', '#e91e8c', '#00b4d8', '#aaaaaa'],
        text=[f"{by_color[c]['ARI']:.3f}" for c in colors_order],
        textposition='outside',
        name='ARI',
    ), row=2, col=1)

    # Target line for ARI bar / ARI 막대 목표선
    # Using xref/yref directly to avoid PlotlyKeyError on bar subplots
    # bar 타입 서브플롯에서 row/col 인자가 PlotlyKeyError 를 발생시키므로 xref/yref 직접 사용
    fig.add_shape(
        type='line', xref='x4', yref='y4',
        x0=-0.5, x1=3.5,
        y0=g['ARI_threshold'], y1=g['ARI_threshold'],
        line=dict(color='#ff7aa2', width=2, dash='dot'),
    )
    fig.add_annotation(
        xref='x4', yref='y4', x=3.2, y=g['ARI_threshold'],
        text=f'Target {g["ARI_threshold"]}',
        showarrow=False, font=dict(color='#ff7aa2', size=11),
        xanchor='right', yanchor='bottom',
    )

    fig.update_layout(
        title='Embedding Quality Metrics Dashboard / Embedding 품질 지표 대시보드',
        title_font=dict(size=17),
        template='plotly_dark',
        font=dict(family='Segoe UI', size=12),
        width=1100, height=700,
        margin=dict(l=40, r=40, t=80, b=40),
        showlegend=False,
    )

    if output_path:
        fig.write_html(output_path, include_plotlyjs='cdn')
        print(f'Saved / 저장: {output_path}')

    return fig


dashboard_path = str(OUTPUT_DIR / 'metrics_dashboard.html')
plot_metrics_dashboard(metrics_summary, output_path=dashboard_path)
open_in_browser(OUTPUT_DIR / 'metrics_dashboard.html')

Saved / 저장: /Users/yangjin-hyeong/Desktop/CMYK_MAIN/data_set/embedding_viz/metrics_dashboard.html


---
## 12. Output Summary / 산출물 목록

In [15]:
print('=' * 60)
print('Generated Outputs / 생성된 산출물 목록')
print('=' * 60)

output_files = [
    ('tsne_by_level.html',         't-SNE Level-colored (interactive)'),
    ('tsne_by_color.html',         't-SNE CMYK channel-colored (interactive)'),
    ('tsne_per_color_subplot.html','t-SNE CMYK 4-panel subplot'),
    ('tsne_mismatch_overlay.html', 't-SNE mismatch samples highlighted'),
    ('umap_by_level.html',         'UMAP Level-colored (requires umap-learn)'),
    ('metrics_dashboard.html',     'ARI / Silhouette / Mismatch dashboard'),
    ('metrics_summary.json',       'ARI / Silhouette / Mismatch numeric summary'),
    ('mismatch_samples.csv',       'Label-embedding mismatch sample list'),
]

for fname, desc in output_files:
    status = '[OK]' if (OUTPUT_DIR / fname).exists() else '[  ]'
    print(f'  {status}  {fname:<42} {desc}')

print()
print('Next steps / 다음 단계 (PRD 3.2.2):')
print('  1. Open tsne_by_level.html and visually inspect Level cluster boundaries')
print('     tsne_by_level.html 열고 Level 군집 경계를 육안으로 확인')
print('  2. Review mismatch_samples.csv and correct labels')
print('     mismatch_samples.csv 검토 후 라벨 수정')
print('  3. Check ARI / mismatch ratio in metrics_dashboard.html')
print('     metrics_dashboard.html 에서 ARI / 불일치 비율 확인')
print('  4. If not met -> correct labels and re-run this notebook (Phase 1 loop)')
print('     기준 미달 -> 라벨 수정 후 이 노트북 재실행 (Phase 1 루프)')
print('  5. If met -> proceed to 03_training.ipynb (Phase 2)')
print('     기준 충족 -> 03_training.ipynb 진행 (Phase 2)')

Generated Outputs / 생성된 산출물 목록
  [OK]  tsne_by_level.html                         t-SNE Level-colored (interactive)
  [OK]  tsne_by_color.html                         t-SNE CMYK channel-colored (interactive)
  [OK]  tsne_per_color_subplot.html                t-SNE CMYK 4-panel subplot
  [OK]  tsne_mismatch_overlay.html                 t-SNE mismatch samples highlighted
  [OK]  umap_by_level.html                         UMAP Level-colored (requires umap-learn)
  [OK]  metrics_dashboard.html                     ARI / Silhouette / Mismatch dashboard
  [OK]  metrics_summary.json                       ARI / Silhouette / Mismatch numeric summary
  [OK]  mismatch_samples.csv                       Label-embedding mismatch sample list

Next steps / 다음 단계 (PRD 3.2.2):
  1. Open tsne_by_level.html and visually inspect Level cluster boundaries
     tsne_by_level.html 열고 Level 군집 경계를 육안으로 확인
  2. Review mismatch_samples.csv and correct labels
     mismatch_samples.csv 검토 후 라벨 수정
  3. Check ARI / mi